# 03 — Evaluation
**Skin Lesion Classification in Thermal Images**

Este notebook **carrega e analisa** os resultados de um treino já executado por `scripts/train_cv.py` — ele não treina nada. Rode o script primeiro (`python scripts/train_cv.py` para o run completo, com ~5h em Apple Silicon; `--quick-test` para validar o pipeline rapidamente no subconjunto pequeno) e então use este notebook para gerar matrizes de confusão, curvas ROC, curvas de treino do fold 0 e as tabelas comparativas finais (accuracy, F1-score, AUC-ROC) entre ThermalCNN, ResNet18, SVM e Random Forest.

## 1. Configuração do run e visão geral do dataset

In [ ]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.dataset import CLASSES, _build_loaders_from_splits, _collect_samples, _patient_kfold_split
from src.evaluate import plot_confusion_matrix, plot_roc_curve, plot_training_curves

RUNS_DIR = Path("../results/runs")

# Nome da pasta em results/runs/ a analisar. Deixe None para pegar a mais
# recente automaticamente, ou defina explicitamente (ex.: "20260707_153200")
# para fixar qual run está sendo citado na análise — importante para
# reprodutibilidade ao escrever o TCC.
RUN_NAME = None

# Só considera pastas no formato atual (com train_config.json) — results/runs/
# também guarda runs "legacy_*" pré-refactor (split único, sem k-fold), que não
# têm os artefatos que este notebook espera (predictions_*.npz, metrics_cv_folds*.csv
# etc.) e não devem ser escolhidas automaticamente como "a mais recente".
valid_runs = [
    p for p in (RUNS_DIR.iterdir() if RUNS_DIR.is_dir() else [])
    if p.is_dir() and (p / "train_config.json").exists()
]
if not valid_runs:
    raise FileNotFoundError(
        f"Nenhuma run no formato atual encontrada em {RUNS_DIR.resolve()}. Rode o treino primeiro, por exemplo:\n"
        f'  python scripts/train_cv.py --quick-test --notes "..."   # validação rápida\n'
        f'  python scripts/train_cv.py --notes "..."                # run completo (~5h)'
    )

if RUN_NAME is None:
    RUN_NAME = max(valid_runs, key=lambda p: p.stat().st_mtime).name

RESULTS_DIR = RUNS_DIR / RUN_NAME
print(f"Analisando run: {RUN_NAME}\n")
print((RESULTS_DIR / "RUN.md").read_text())

with open(RESULTS_DIR / "train_config.json") as f:
    train_config = json.load(f)

In [ ]:
# Recomputa (não retreina) a composição dos folds a partir da config salva pelo
# script de treino — percorrer o filesystem e refazer o split por paciente é
# barato (segundos), ao contrário do treino em si.
DATA_ROOT = Path(train_config["data_root"])
FRAME_STRIDE = train_config["frame_stride"]
K_FOLDS = train_config["k_folds"]

samples = _collect_samples(DATA_ROOT, frame_stride=FRAME_STRIDE)
folds = _patient_kfold_split(samples, k=K_FOLDS, seed=train_config["seed"])

print(f"Total de amostras (após stride={FRAME_STRIDE}): {len(samples):,}")
print(f"\n{'Fold':<6} {'Train':>8} {'Val':>8} {'Test':>8} {'TrainP':>8} {'ValP':>6} {'TestP':>6}")
print("-" * 54)
for i, f in enumerate(folds):
    n_train_p = len(set(s["patient_id"] for s in f["train"]))
    n_val_p   = len(set(s["patient_id"] for s in f["val"]))
    n_test_p  = len(set(s["patient_id"] for s in f["test"]))
    print(f"{i:<6} {len(f['train']):>8,} {len(f['val']):>8,} {len(f['test']):>8,} "
          f"{n_train_p:>8} {n_val_p:>6} {n_test_p:>6}")

In [ ]:
# Constrói loaders só do fold 0, para inspeção rápida (shape, normalização, batch de exemplo).
# Os loaders "de verdade" usados no treino são construídos por fold, dentro do loop de CV abaixo.
train_loader, val_loader, test_loader, mean, std = _build_loaders_from_splits(
    folds[0], batch_size=32, num_workers=0,
)
print(f"Média treino (fold 0):  {mean:.4f}")
print(f"Desvio treino (fold 0): {std:.4f}")

# Verifica shape e intervalo de valores de um batch
images, labels = next(iter(train_loader))
print(f"Shape do batch:  {tuple(images.shape)}")   # esperado: (32, 1, 224, 224)
print(f"Dtype:           {images.dtype}")
print(f"Valores min/max: {images.min():.3f} / {images.max():.3f}")
print(f"Labels únicos:   {labels.unique().tolist()}")

In [ ]:
# Visualiza 8 frames do batch com seus labels
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
fig.suptitle("Amostras do batch de treino (após normalização → desnormalizado para exibição)", fontsize=11)

for i, ax in enumerate(axes.flat):
    img = images[i, 0]  # canal único
    img_display = img * std + mean  # desnormaliza para [0, 1]
    ax.imshow(img_display.numpy(), cmap="gray", vmin=0, vmax=1)
    ax.set_title(CLASSES[labels[i].item()], fontsize=10)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 2. Curvas de treino (fold 0)

Carregadas do histórico salvo por `scripts/train_cv.py` (`history_*_fold0.json`) —
diagnóstico do fold 0 apenas, não a métrica final (essa vem da agregação entre
folds nas seções seguintes).

In [ ]:
def load_history(model_slug):
    with open(RESULTS_DIR / f"history_{model_slug}_fold0.json") as f:
        return json.load(f)

thermal_cnn_history = load_history("thermal_cnn")
resnet18_history = load_history("resnet18")

plot_training_curves(
    thermal_cnn_history["train_loss"], thermal_cnn_history["val_loss"],
    thermal_cnn_history["train_acc"],  thermal_cnn_history["val_acc"],
    save_path=RESULTS_DIR / "curves_thermal_cnn_fold0.png",
)
plot_training_curves(
    resnet18_history["train_loss"], resnet18_history["val_loss"],
    resnet18_history["train_acc"],  resnet18_history["val_acc"],
    save_path=RESULTS_DIR / "curves_resnet18_fold0.png",
)

## 3. Resultados agregados (predições combinadas de todos os folds)

Matrizes de confusão e curvas ROC construídas a partir das predições de **todos** os
folds de teste combinados (cada paciente contribui exatamente uma vez, no fold em que
foi usado como teste), carregadas de `results/predictions_*.npz` — dão uma visão geral
do comportamento do modelo no dataset completo, complementar à média±desvio por fold
da Seção 4.

In [ ]:
MODEL_NAMES = ["ThermalCNN", "ResNet18", "SVM", "Random Forest"]
FILE_SLUG = {"ThermalCNN": "thermal_cnn", "ResNet18": "resnet18", "SVM": "svm", "Random Forest": "rf"}

# Predições consolidadas de todos os folds (probabilidades empilhadas por linha),
# salvas por scripts/train_cv.py — nada é recalculado aqui.
pooled = {}
for name in MODEL_NAMES:
    data = np.load(RESULTS_DIR / f"predictions_{FILE_SLUG[name]}.npz")
    pooled[name] = (data["y_true"], data["y_pred"], data["y_prob"])
    print(f"{name:<15} y_true={data['y_true'].shape}  y_prob={data['y_prob'].shape}")

In [ ]:
# Matrizes de confusão (predições agregadas de todos os folds)
FILE_SLUG = {"ThermalCNN": "thermal_cnn", "ResNet18": "resnet18", "SVM": "svm", "Random Forest": "rf"}

for name in MODEL_NAMES:
    y_true, y_pred, _ = pooled[name]
    plot_confusion_matrix(y_true, y_pred, CLASSES, save_path=RESULTS_DIR / f"cm_{FILE_SLUG[name]}.png")

In [ ]:
# Curvas ROC (predições agregadas de todos os folds)
for name in MODEL_NAMES:
    y_true, _, y_prob = pooled[name]
    plot_roc_curve(y_true, y_prob, CLASSES, save_path=RESULTS_DIR / f"roc_{FILE_SLUG[name]}.png")

In [ ]:
# Métricas por fold (nível frame e nível paciente/sequência), geradas por
# scripts/train_cv.py — carregadas, não recalculadas.
folds_df = pd.read_csv(RESULTS_DIR / "metrics_cv_folds.csv")
folds_seq_df = pd.read_csv(RESULTS_DIR / "metrics_cv_folds_patient.csv")
folds_df

## 4. Tabela comparativa final (média ± desvio padrão entre os folds)

In [ ]:
metric_cols = ["accuracy", "precision", "recall", "specificity", "f1", "auc_roc"]

def summarize(df):
    summary = (
        df.groupby("model")[metric_cols]
        .agg(["mean", "std"])
        .reindex(MODEL_NAMES)
    )
    # Achata o índice de colunas multi-nível em "métrica (mean/std)"
    summary.columns = [f"{metric} ({stat})" for metric, stat in summary.columns]
    return summary.round(4)

summary = summarize(folds_df)
summary_csv = RESULTS_DIR / "metrics.csv"
summary.to_csv(summary_csv)
print(f"Resumo nível frame (média±desvio) salvo em {summary_csv}")
summary

## 5. Tabela comparativa — nível paciente/sequência (agregado)

Mesmas métricas da Seção 4, mas com as predições agregadas por sequência
(patient_id + classe) antes de calcular a métrica — média das probabilidades
softmax de todos os frames de teste de uma mesma sequência, seguida de argmax.
Frames vizinhos de uma sequência térmica são quase idênticos entre si, então essa
agregação deveria reduzir o ruído de frame individual e a variância entre folds
(comparar o desvio-padrão desta tabela com o da Seção 4).

In [ ]:
summary_seq = summarize(folds_seq_df)
summary_seq_csv = RESULTS_DIR / "metrics_patient.csv"
summary_seq.to_csv(summary_seq_csv)
print(f"Resumo nível paciente (média±desvio) salvo em {summary_seq_csv}")
summary_seq